In [1]:
def classification_metrics(matrix, labels):
    n = len(labels)
    per_class = {}

    for i, label in enumerate(labels):
        tp = matrix[i][i]
        predicted_total = sum(matrix[i])       # row total
        gold_total = sum(matrix[r][i] for r in range(n))  # column total
        precision = tp / predicted_total if predicted_total else 0.0
        recall = tp / gold_total if gold_total else 0.0
        per_class[label] = (precision, recall)

    macro_p = sum(v[0] for v in per_class.values()) / n
    macro_r = sum(v[1] for v in per_class.values()) / n

    total_tp = sum(matrix[i][i] for i in range(n))
    total_predictions = sum(sum(row) for row in matrix)
    total_gold = total_predictions
    micro_p = total_tp / total_predictions
    micro_r = total_tp / total_gold

    return per_class, macro_p, macro_r, micro_p, micro_r


matrix = [
    [5, 10, 5],
    [15, 20, 10],
    [0, 15, 10],
]
labels = ["Cat", "Dog", "Rabbit"]

per_class, macro_p, macro_r, micro_p, micro_r =     classification_metrics(matrix, labels)

for label, (precision, recall) in per_class.items():
    print(f"{label}: precision={precision:.4f}, recall={recall:.4f}")
print(f"Macro: precision={macro_p:.4f}, recall={macro_r:.4f}")
print(f"Micro: precision={micro_p:.4f}, recall={micro_r:.4f}")

Cat: precision=0.2500, recall=0.2500
Dog: precision=0.4444, recall=0.4444
Rabbit: precision=0.4000, recall=0.4000
Macro: precision=0.3648, recall=0.3648
Micro: precision=0.3889, recall=0.3889


In [3]:
from collections import Counter
from math import prod

corpus = [
    "<s> I love NLP </s>".split(),
    "<s> I love deep learning </s>".split(),
    "<s> deep learning is fun </s>".split(),
]

unigram_counts = Counter()
bigram_counts = Counter()

for sentence in corpus:
    unigram_counts.update(sentence)
    bigram_counts.update(zip(sentence, sentence[1:]))


def bigram_probability(previous_word, next_word):
    return bigram_counts[(previous_word, next_word)] / unigram_counts[previous_word]


def sentence_probability(sentence):
    tokens = sentence.split() if isinstance(sentence, str) else sentence
    probabilities = []
    for previous_word, next_word in zip(tokens, tokens[1:]):
        denominator = unigram_counts[previous_word]
        if denominator == 0:
            return 0.0
        probabilities.append(
            bigram_counts[(previous_word, next_word)] / denominator
        )
    return prod(probabilities)


s1 = "<s> I love NLP </s>"
s2 = "<s> I love deep learning </s>"
p1 = sentence_probability(s1)
p2 = sentence_probability(s2)

print("Unigram counts:", dict(unigram_counts))
print("Bigram counts:", dict(bigram_counts))
print(f"P(S1) = {p1:.6f}")
print(f"P(S2) = {p2:.6f}")

preferred = "S1" if p1 > p2 else "S2" if p2 > p1 else "Neither; they tie"
print( preferred)
print("S1 has the larger product of MLE bigram probabilities.")


Unigram counts: {'<s>': 3, 'I': 2, 'love': 2, 'NLP': 1, '</s>': 3, 'deep': 2, 'learning': 2, 'is': 1, 'fun': 1}
Bigram counts: {('<s>', 'I'): 2, ('I', 'love'): 2, ('love', 'NLP'): 1, ('NLP', '</s>'): 1, ('love', 'deep'): 1, ('deep', 'learning'): 2, ('learning', '</s>'): 1, ('<s>', 'deep'): 1, ('learning', 'is'): 1, ('is', 'fun'): 1, ('fun', '</s>'): 1}
P(S1) = 0.333333
P(S2) = 0.166667
S1
S1 has the larger product of MLE bigram probabilities.
